# Notebook 3: Neural Machine Translation with T5

In this notebook we perform **neural machine translation** using Google's T5 (Text-To-Text Transfer Transformer) model via the Hugging Face `transformers` library.

## Learning Objectives
- Understand how sequence-to-sequence (seq2seq) models approach translation
- Use the `pipeline` API for translation tasks
- Manually tokenize and decode with T5's prefix-based task conditioning
- Evaluate translation quality with the **BLEU** metric
- Discuss multi-lingual extensions (mBERT, mT5)

## Background
T5 frames **every NLP task as a text-to-text problem**. For translation, the input is prefixed with `"translate English to German: "` (or other language pairs) and the model generates the translated string. We use `t5-small` (60 M parameters) which can run comfortably on CPU.

## 1. Install & Import Dependencies

In [ ]:
# !pip install transformers torch sentencepiece sacrebleu evaluate

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Translation via the Pipeline API

In [ ]:
# T5-small supports: English → German, French, Romanian
en_de_translator = pipeline(
    "translation_en_to_de",
    model="t5-small",
    device=0 if torch.cuda.is_available() else -1,
)

en_fr_translator = pipeline(
    "translation_en_to_fr",
    model="t5-small",
    device=0 if torch.cuda.is_available() else -1,
)

sentences = [
    "The weather is beautiful today and I feel great.",
    "Machine learning is transforming every industry.",
    "I would like to order a coffee and a croissant, please.",
]

print(f"{'English':<55} {'German':<55} {'French'}")
print("-" * 165)
for sent in sentences:
    de = en_de_translator(sent, max_length=128)[0]["translation_text"]
    fr = en_fr_translator(sent, max_length=128)[0]["translation_text"]
    print(f"{sent:<55} {de:<55} {fr}")

## 3. Low-Level Translation with T5

Under the hood, T5 receives a prefixed string like `"translate English to German: <sentence>"` and generates the target language text.

In [ ]:
MODEL_NAME = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"T5 model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
def translate(text: str, source: str = "English", target: str = "German") -> str:
    """
    Translate text using T5's text-to-text framing.

    Parameters
    ----------
    text   : input text in `source` language
    source : source language name (e.g. 'English')
    target : target language name (e.g. 'German', 'French', 'Romanian')

    Returns
    -------
    str : translated text
    """
    prefix = f"translate {source} to {target}: "
    input_text = prefix + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Test with different language pairs
test_text = "Artificial intelligence is reshaping how we interact with technology."

for target_lang in ["German", "French", "Romanian"]:
    translation = translate(test_text, target=target_lang)
    print(f"[EN → {target_lang:8}] {translation}")

## 4. Evaluating Translation Quality with BLEU

**BLEU** (Bilingual Evaluation Understudy) measures how many n-gram sequences in the hypothesis match n-grams in the reference translation. A score of 1.0 is a perfect match; real-world systems typically score between 0.25 and 0.45.

In [ ]:
try:
    import evaluate
    bleu = evaluate.load("sacrebleu")
    USE_EVALUATE = True
except Exception:
    # Fallback: use sacrebleu directly
    from sacrebleu.metrics import BLEU
    bleu_metric = BLEU()
    USE_EVALUATE = False

# Small reference dataset: (English source, German reference)
en_sentences = [
    "The sun rises in the east and sets in the west.",
    "I enjoy reading books about history and science.",
    "The train arrives at the station every hour.",
    "Please pass the salt and pepper.",
    "Technology has made communication easier than ever before.",
]

# Reference translations (would normally come from human translators)
de_references = [
    "Die Sonne geht im Osten auf und im Westen unter.",
    "Ich lese gerne Bücher über Geschichte und Wissenschaft.",
    "Der Zug kommt jede Stunde am Bahnhof an.",
    "Bitte reiche mir das Salz und den Pfeffer.",
    "Die Technologie hat die Kommunikation einfacher gemacht als je zuvor.",
]

# Generate hypothesis translations
hypotheses = [translate(s, target="German") for s in en_sentences]

print("Source → Hypothesis → Reference")
print("-" * 80)
for src, hyp, ref in zip(en_sentences, hypotheses, de_references):
    print(f"SRC : {src}")
    print(f"HYP : {hyp}")
    print(f"REF : {ref}")
    print()

# Compute corpus-level BLEU
if USE_EVALUATE:
    results = bleu.compute(predictions=hypotheses, references=[[r] for r in de_references])
    print(f"Corpus BLEU score: {results['score']:.2f}")
else:
    result = bleu_metric.corpus_score(hypotheses, [de_references])
    print(f"Corpus BLEU score: {result.score:.2f}")

## 5. Sentence-Level BLEU Analysis

In [ ]:
import matplotlib.pyplot as plt

sentence_bleus = []
for hyp, ref in zip(hypotheses, de_references):
    if USE_EVALUATE:
        score = bleu.compute(predictions=[hyp], references=[[ref]])["score"]
    else:
        score = bleu_metric.sentence_score(hyp, [ref]).score
    sentence_bleus.append(score)

plt.figure(figsize=(10, 4))
plt.bar(range(1, len(sentence_bleus) + 1), sentence_bleus, color="mediumseagreen")
plt.xlabel("Sentence index")
plt.ylabel("BLEU score")
plt.title("Sentence-level BLEU scores (EN → DE, t5-small)")
plt.xticks(range(1, len(sentence_bleus) + 1))
plt.tight_layout()
plt.show()

## 6. Multi-lingual Extension: Helsinki-NLP Models

For better translation quality and broader language coverage, Hugging Face hosts the [Helsinki-NLP/opus-mt-*](https://huggingface.co/Helsinki-NLP) collection — each model specialises in one language pair.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Helsinki-NLP English → Spanish
MARIAN_MODEL = "Helsinki-NLP/opus-mt-en-es"
marian_tokenizer = MarianTokenizer.from_pretrained(MARIAN_MODEL)
marian_model = MarianMTModel.from_pretrained(MARIAN_MODEL).to(device)
marian_model.eval()

def translate_marian(texts: list[str]) -> list[str]:
    """Translate a list of English sentences to Spanish with MarianMT."""
    # MarianMT expects a special format: >>tgt_lang<< prefix is optional for single-pair models
    inputs = marian_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)
    with torch.no_grad():
        output_ids = marian_model.generate(**inputs, num_beams=4, max_new_tokens=128)
    return [marian_tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]


en_texts = [
    "Hello, how are you doing today?",
    "The library closes at nine o'clock in the evening.",
    "I would like a table for two, please.",
]

es_translations = translate_marian(en_texts)

print("English → Spanish (Helsinki-NLP/opus-mt-en-es)")
print("-" * 60)
for en, es in zip(en_texts, es_translations):
    print(f"EN: {en}")
    print(f"ES: {es}")
    print()

## 7. Summary

In this notebook we:
- Used T5's text-to-text framing for English → German/French/Romanian translation
- Computed BLEU scores to evaluate translation quality
- Explored Helsinki-NLP MarianMT models for English → Spanish translation

**Extension Ideas**
- Fine-tune an mT5 or mBART model on a domain-specific parallel corpus
- Add more languages using the Helsinki-NLP model zoo
- Build a Gradio app with language selection dropdowns

**Next**: `04_question_answering.ipynb` — extractive and generative QA.